# Fused MPO

In the quantum chemistry code I introduced a different kind of MPOHamiltonian type. In this notebook I will try to motivate that it is a good idea.

In [1]:
using LinearAlgebra, MPSKit, TensorKit, MPSKitExperimental
LinearAlgebra.BLAS.set_num_threads(1);

In [2]:
include("opsum_qchem_su2.jl"); using .OpSumQChemSU2

In [3]:
file = "N2.CCPVDZ.FCIDUMP"

# ERI is the Electronic Repulsion Integral, K is the two-body term, E0 is an energy shift,
# NORB is the number of orbitals (mps length), NELEC is the number of electrons and MS2 is the total spin.
# fcidump files can contain orbital symmetries, but these are ignored.
(ERI, K, E0, NORB, NELEC, MS2) = parse_fcidump(file);

In [4]:
D = 50          # the OpSum MPO uses in-RAM environments; D = 200 likely doesn't fit in 31 GB
sweeps = 10

10

## DMRG comparison

Opsum is a package that is able to construct the quantum chemistry hamiltonian automatically, and optimally, as a conventional MPOHamiltonian.

In [5]:
# same input as the fused MPO below; E0 included
t_opsum = @elapsed ham_opsum = opsum_su2_hamiltonian(E0, K, ERI)
println("OpSum MPO built in $(round(t_opsum; digits = 1)) s")
println("bond dims (dense): ", [dim(right_virtualspace(ham_opsum, i)) for i in 1:NORB])

OpSum MPO built in 51.7 s
bond dims (dense): [17, 128, 263, 462, 725, 1060, 1451, 1914, 2481, 3064, 3711, 4422, 5197, 5924, 5197, 4422, 3711, 3064, 2481, 1962, 1507, 1052, 725, 462, 239, 104, 17, 1]


In [6]:
t_fused = @elapsed ham = quantum_chemistry_hamiltonian(E0, K, ERI);
println("fused MPO built in $(round(t_fused; digits = 1)) s")

fused MPO built in 255.6 s


┌ Warning: JSON-RPC endpoint has been blocked writing to its peer; outgoing messages are queueing up and will not be delivered until the peer resumes reading
│   blocked_seconds = 255.6
│   queued_messages = 1
└ @ VSCodeServer.JSONRPC /home/maarten/.vscode-oss/extensions/julialang.language-julia-1.243.2-universal/scripts/packages/JSONRPC/src/core.jl:414


In [7]:
psp = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((0, 0, 0) => 1, (1, 1 // 2, 1) => 1, (2, 0, 0) => 1);

left = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((-NELEC, MS2 // 2, mod(-NELEC, 2)) => 1);
right = oneunit(left);

virtual = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((i, s, b) => 1 for i in -NELEC:0, s in 0:(1 // 2):(MS2 // 2 + 1), b in (0, 1));

st = FiniteMPS(rand, Float64, NORB, psp, virtual; left, right);
st0 = copy(st);   # same starting state for both DMRG runs

In [8]:
# Same operator? ⟨st|H|st⟩ on the random starting state (not a ground state, so the value
# itself is meaningless; only the difference matters, expect ~1e-12).
e_fused = real(expectation_value(st, ham, disk_environments(st, ham)))
e_opsum = real(expectation_value(st, ham_opsum))
println("⟨H_opsum⟩ - ⟨H_fused⟩ on a random MPS: ", e_opsum - e_fused)

⟨H_opsum⟩ - ⟨H_fused⟩ on a random MPS: 9.521272659185342e-13


In [9]:

# environments that live on disk, instead of in ram
envs = disk_environments(st, ham);

In [10]:

(st, envs) = find_groundstate(st, ham, DMRG2(; trscheme = truncrank(D), verbosity = 1000, maxiter = sweeps, miniter = sweeps), envs);

println("E = ", real(expectation_value(st, ham, envs)))


┌ Info: DMRG2   1:	obj = -1.073075229215e+02	err = 9.9999993517e-01	time = 1.10 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   2:	obj = -1.076696458779e+02	err = 9.9793075025e-01	time = 28.83 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   3:	obj = -1.084507396810e+02	err = 9.9997360519e-01	time = 29.60 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   4:	obj = -1.090688517896e+02	err = 2.5070059141e-01	time = 28.50 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   5:	obj = -1.091315763106e+02	err = 6.3688924884e-03	time = 26.07 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   6:	obj = -1.091387882971e+02	err = 3.4637920202e-04	time = 27.89 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.

E = -109.16554802372582


┌ Warning: DMRG2 cancel 10:	obj = -1.091655480237e+02	err = 2.4757637499e-04	time = 5.67 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:195


In [11]:
(st_opsum, envs_opsum) = find_groundstate(copy(st0), ham_opsum, DMRG2(; trscheme = truncrank(D), verbosity = 1000, maxiter = sweeps, miniter = sweeps));

E_opsum = real(expectation_value(st_opsum, ham_opsum, envs_opsum))
E_fused = real(expectation_value(st, ham, envs))
println("E (OpSum) = ", E_opsum, "    E (fused) = ", E_fused, "    diff = ", E_opsum - E_fused)

┌ Warning: JSON-RPC endpoint has been blocked writing to its peer; outgoing messages are queueing up and will not be delivered until the peer resumes reading
│   blocked_seconds = 150.7
│   queued_messages = 0
└ @ VSCodeServer.JSONRPC /home/maarten/.vscode-oss/extensions/julialang.language-julia-1.243.2-universal/scripts/packages/JSONRPC/src/core.jl:414
┌ Info: DMRG2   1:	obj = -1.073075168882e+02	err = 9.9999993517e-01	time = 4.97 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   2:	obj = -1.076696464134e+02	err = 9.9793097034e-01	time = 2.96 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   3:	obj = -1.084507239950e+02	err = 9.9997379399e-01	time = 3.00 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   4:	obj = -1.090688315404e+02	err = 2.5114026334e-01	time = 2.92 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/

E (OpSum) = -109.1655435154882    E (fused) = -109.16554802372582    diff = 4.508237623213063e-6


## How it works

`LinkMPOHamiltonian` represents an mpo as

$$H = b_L\,\mathrm{diag}(O_1)\,C_1\,\mathrm{diag}(O_2)\,C_2 \cdots C_{N-1}\,\mathrm{diag}(O_N)\,b_R .$$

Hierin zijn $O_{n,k}$ de *kanalen* van site $n$ (één operator elk) en $C_n$ scalaire matrices tussen de kanalen van naburige sites. Een gewone MPO is het speciale geval waarin je elke $C_n = X_n Y_n$ op één manier factoriseert en de factoren in de operatoren vermenigvuldigt: $W_n = Y_n\,\mathrm{diag}(O_n)\,X_{n+1}$. Hier is die factorisatie afgeleid: ze bepaalt alleen op welke "bondtoestanden" de omgevingen bewaard worden, en wordt gekozen via een minimale vertex cover van het patroon van $C_n$ (König).

### Voorbeeld

Hard-core bosonen met naaste-buren-hopping en een **dichte** lange-dracht-interactie:

$$H = -\sum_j t_j\,(b^\dagger_j b_{j+1} + h.c.) + \sum_i \mu_i n_i + \sum_{i<j} V_{ij}\, n_i n_j .$$

De kanalen op een site zijn $1, n, b^\dagger, b$. Voor $V$ gebruik ik dezelfde boekhouding als in kwantumchemie (`examples/link_example.jl`):
- **links van de switch** onthoudt een bond welke $n_i$ al geplaatst zijn (normaal, $k$ toestanden);
- **rechts ervan** welke complementaire operator $Q_m = \sum_i V_{im} n_i$ nog gesloten moet worden ($N-k$ toestanden);
- **op de switch** zet een identiteitskanaal het ene om in het andere: de link daar bevat het dichte blok $V_{im}$.

Alle coëfficiënten $(t, \mu, V)$ staan in de links.


In [12]:
include("link_example.jl")
using Random; Random.seed!(3)
N = 10; Np = 5
t = 1 .+ 0.2rand(N - 1); μ = randn(N); V = (A = rand(N, N); (A + A') / 2)   # V is dense
ops, links = link_hamiltonian(t, μ, V)
h = LinkMPOHamiltonian(ops, links, fill(P, N))
println(h)
for n in 1:N+1
    println("link $n: C is ", join(size(h.links[n]), "×"), " with ", nnz(h.links[n]), " entries, ", length(h.bondspaces[n]), " bond states")
end

LinkMPOHamiltonian on 10 sites: 104 channels, 175 link entries, 63 bond states
link 1: C is 1×6 with 5 entries, 1 bond states
link 2: C is 6×10 with 10 entries, 5 bond states
link 3: C is 10×11 with 16 entries, 6 bond states
link 4: C is 11×12 with 18 entries, 7 bond states
link 5: C is 12×13 with 20 entries, 8 bond states
link 6: C is 13×13 with 37 entries, 9 bond states
link 7: C is 13×12 with 20 entries, 8 bond states
link 8: C is 12×11 with 18 entries, 7 bond states
link 9: C is 11×10 with 16 entries, 6 bond states
link 10: C is 10×6 with 10 entries, 5 bond states
link 11: C is 6×1 with 5 entries, 1 bond states


Uitvoer (2026-10-02): link 6 is $13\times 13$ met 37 elementen, waarvan het dichte $V$-blok $5\times 4$ is (de switch). Het aantal bondtoestanden is $1, 5, 6, 7, 8, 9, 8, 7, 6, 5, 1$: de vertex cover kiest vanzelf de kleinste kant, $\min(k, N-k)$ plus de vaste toestanden.

### Waar het loont

Tel per site en per link, met $k$ normale en $m$ complementaire kanalen rond de switch, $D$ de MPS-bond en $d$ de fysische dimensie.

1. **De switch is scalair.** Hier staat het $k\times m$-blok $V$ in $C$. In een gewone MPO zit het in $W$ als $k\,m$ operatorwaardige elementen $V_{im}\cdot 1$.
   - *Omgevingsupdate door de switch:* in de linkvorm $k\,m$ axpy's op omgevingen ($D^2$ elk) plus één contractie per kanaal. Een gewone MPO doet $k\,m$ operatortoepassingen ($D^2 d^2$ elk), en wie dat wil vermijden heeft block2's tweede MPO plus `MPOSchemer` nodig.
   - *AC2 rond de switch:* $C$ wordt één keer per eigsolve gefactoriseerd tot $\operatorname{rang} = \min(k,m)$ termen. Een gewone MPO in de gebruikelijke volgorde doet de $k\,m$ operatortoepassingen bij *elke* matvec.
   - *Wanneer het telt:* voor kwantumchemie zijn $k, m \sim N^2/4$ paren, dus $k\,m \sim N^4/16$ tegenover $\sim N^2 D$ voor de omgevingen. Dat telt zodra $N^2 \gtrsim D$.
2. **De omgevingsbasis is vrij.** Een gewone MPO legt één factorisatie per bond vast. Hier komt ze uit $C$, met het minimale aantal bondtoestanden, zonder handwerk: 279 tegenover 310 voor STO-3G en 4510 tegenover 4973 voor cc-pVDZ met de handgebouwde MPO.
3. **De AC2-matvec is die van MPSKit, met dezelfde stapel.** `link_AC2_hamiltonian` bouwt MPSKits voorberekende vorm: $GL\cdot O$ en $O\cdot GR$, gestapeld over de middelste bondtoestanden, waarbij de omgevingen eerst met de linkscalairen gecombineerd worden en pas daarna één keer per operator gecontraheerd. De matvec is dan dezelfde handvol blok-GEMMs als bij MPSKit. Rangreductie van $C$ per sectorpatroon (de vroegere tabelvorm) bespaart flops, maar splitst de matvec in duizenden kleine GEMMs, wat bij $D=50$–$100$ meer kost dan het oplevert; en per middelste ruimte heeft $X\cdot Y$ al de rang van $C$, dus de stapel kan niet korter.
4. **Parameters in de links: afgeleiden gratis.** $E = \langle H\rangle$ is lineair in elke link, met
   $$\frac{\partial E}{\partial C_n[k,k']} = \big\langle\text{alles links dat eindigt in kanaal } k \,\big|\, \text{alles rechts dat begint in kanaal } k'\big\rangle .$$
   Dat is één paring per element, uit één links-rechts-sweep (`link_gradient`). Staan alle parameters $\theta$ lineair in de links, dan is $\partial E/\partial\theta = \sum G\cdot(\text{coëfficiënt})$.
   - *In dit voorbeeld* geeft dat $\langle n_i\rangle$, $\langle n_i n_j\rangle$ en $\langle b^\dagger_j b_{j+1} + h.c.\rangle$ zonder extra code.
   - *In kwantumchemie* zijn het de 1- en 2-RDM's: `qchem_rdms` is nu precies dit en vervangt ~1000 regels handgeschreven contracties. Het resultaat is identiek tot op $3\cdot 10^{-15}$, en op cc-pVDZ ($D=30$) duurt het 3.4 s in plaats van 5.0 s.


In [13]:
ψ0 = FiniteMPS(rand, Float64, N, P, U1Space((c => 64 for c in -Np:0)...); left = U1Space(-Np => 1), right = U1Space(0 => 1))
ψ, _ = find_groundstate(ψ0, h, DMRG2(; trscheme = truncrank(256), maxiter = 10, verbosity = 0))
ψ, envs = find_groundstate(ψ, h, DMRG(; maxiter = 10, verbosity = 0))
E_ed, hop, nexp, nnexp = ed(t, μ, V, Np)                      # exact diagonalization, N = 10

ts, μs, Vs, np = parameters(N)                                 # the same channels with symbolic coefficients
sops, slinks = link_hamiltonian(ts, μs, Vs)
g, _ = link_gradient(normalize(ψ), MX.gradient_hamiltonian(sops, slinks, fill(P, N)), slinks; nparams = np)
pairs = [(i, j) for i in 1:N for j in i+1:N]
println("E - E_ED = ", real(expectation_value(ψ, h, envs)) - E_ed)
println("max |∂E/∂t_j + ⟨b†_j b_j+1 + h.c.⟩| = ", maximum(abs, g[1:N-1] .+ hop))
println("max |∂E/∂μ_i - ⟨n_i⟩|              = ", maximum(abs, g[N:2N-1] .- nexp))
println("max |∂E/∂V_ij - ⟨n_i n_j⟩|         = ", maximum(abs, [g[2N - 1 + k] - nnexp[i, j] for (k, (i, j)) in enumerate(pairs)]))

E - E_ED = 1.687538997430238e-14
max |∂E/∂t_j + ⟨b†_j b_j+1 + h.c.⟩| = 1.1712852909795402e-13
max |∂E/∂μ_i - ⟨n_i⟩|              = 1.021405182655144e-13
max |∂E/∂V_ij - ⟨n_i n_j⟩|         = 1.1307621505807219e-13


Uitvoer (2026-10-02): $E - E_{ED} = 3\cdot 10^{-15}$, en alle drie de afgeleiden zijn gelijk aan de ED-verwachtingswaarden tot op $1.2\cdot 10^{-13}$.

### Wanneer het niet loont

De AC2-matvec zelf heeft minstens zoveel termen als de operator-Schmidtrang over de middelste bond. Een gewone MPO met dezelfde factorisatie op die bond haalt dezelfde matvec; dat heb ik nagerekend, het geeft dezelfde tabel. Zonder symmetrie (geen letters) en zonder dichte junctie (geen switch) is de linkvorm dus precies een gewone MPO.

De winst zit in:
- de omgevingen (scalaire junctie, minimale basis zonder handwerk);
- de vrijheid om de matvec-termen los van de omgevingen te kiezen;
- dat parameters data in de links zijn, waardoor afgeleiden (RDM's) en nieuwe integralen een evaluatie zijn in plaats van nieuwe code.
